In [8]:
import pickle
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== 数据集 =====
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ===== 加载数据 =====
with open('data/20news_split.pkl', 'rb') as f:
    dataset = pickle.load(f)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = NewsDataset(*dataset['train'], tokenizer)
val_dataset = NewsDataset(*dataset['val'], tokenizer)
test_dataset = NewsDataset(*dataset['test'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# ===== 模型 =====
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

# ===== 最优模型记录 =====
best_acc = 0

# ===== 评估 =====
def evaluate(loader):
    model.eval()
    preds, labels_list = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    acc = accuracy_score(labels_list, preds)
    print(f"Accuracy: {acc:.4f}")   # ✅ 打印
    return acc                     # ✅ 返回

# ===== 训练 =====
def train():
    global best_acc

    for epoch in range(3):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"\nEpoch {epoch+1}, Loss: {total_loss:.4f}")

        # ===== 验证 =====
        val_acc = evaluate(val_loader)

        # ===== 保存最佳模型 =====
        if val_acc > best_acc:
            best_acc = val_acc
            os.makedirs('model', exist_ok=True)
            torch.save(model.state_dict(), 'model/best_bert.pt')
            print("✔ 已保存最佳模型")

# ===== 测试 =====
def test():
    model.load_state_dict(torch.load( 'model/best_bert.pt'))
    print("\nTest:")
    evaluate(test_loader)
  
# ===== 主函数 =====
if __name__ == "__main__":
    train()
    test()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Epoch 1, Loss: 77.1380
Accuracy: 0.8704
✔ 已保存最佳模型

Epoch 2, Loss: 21.0678
Accuracy: 0.9753
✔ 已保存最佳模型

Epoch 3, Loss: 9.9495
Accuracy: 0.9691

Test:
Accuracy: 0.9778
